conexión con sentinel-2 y datos


In [1]:
%pip install sentinelhub

   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ------------------ --------------------- 2.9/6.3 MB 15.2 MB/s eta 0:00:01
   ------------------------------- -------- 5.0/6.3 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 11.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 18.7 MB/s  0:00:00

   ----------------------------------------  0/14 [aenum]
   ----- ----------------------------------  2/14 [tomli-w]
   ----------- ----------------------------  4/14 [tifffile]
   ----------- ----------------------------  4/14 [tifffile]
   ----------- ----------------------------  4/14 [tifffile]
   -------------- -------------------------  5/14 [shapely]
   -------------- -------------------------  5/14 [shapely]
   -------------- -------------------------  5/14 [shapely]
   -------------- -------------------------  5/14 [shapely]
   -------------- 


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install rasterio

   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.7 MB ? eta -:--:--
   --- ------------------------------------ 2.4/25.7 MB 9.0 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/25.7 MB 11.8 MB/s eta 0:00:02
   ------------ --------------------------- 7.9/25.7 MB 11.3 MB/s eta 0:00:02
   ----------------- ---------------------- 11.5/25.7 MB 12.4 MB/s eta 0:00:02
   ----------------------- ---------------- 15.2/25.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------- ----------- 18.6/25.7 MB 14.3 MB/s eta 0:00:01
   ----------------------------------- ---- 23.1/25.7 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------  25.7/25.7 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------- 25.7/25.7 MB 13.6 MB/s  0:00:02

  Attempting uninstall: click

    Found existing installation: click 8.2.1

    Uninstalling click-8.2.1:

      Successfully uninstalled click-8.2.1

   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.2 which is incompatible.

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from datetime import datetime

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    SentinelHubRequest,
    MimeType,
    bbox_to_dimensions
)

# para datos espaciales
import rasterio
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt


In [4]:
# vamos a usar la config de defecto por mientras
config = SHConfig()
print(f"Configuración de Sentinel Hub: {config}" )

Configuración de Sentinel Hub: {
  "instance_id": "",
  "sh_client_id": "",
  "sh_client_secret": "",
  "sh_base_url": "https://services.sentinel-hub.com",
  "sh_auth_base_url": null,
  "sh_token_url": "https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token",
  "geopedia_wms_url": "https://service.geopedia.world",
  "geopedia_rest_url": "https://www.geopedia.world/rest",
  "aws_access_key_id": "",
  "aws_secret_access_key": "",
  "aws_session_token": "",
  "aws_metadata_url": "https://roda.sentinel-hub.com",
  "aws_s3_l1c_bucket": "sentinel-s2-l1c",
  "aws_s3_l2a_bucket": "sentinel-s2-l2a",
  "opensearch_url": "http://opensearch.sentinel-hub.com/resto/api/collections/Sentinel2",
  "max_wfs_records_per_query": 100,
  "max_opensearch_records_per_query": 500,
  "max_download_attempts": 4,
  "download_sleep_time": 5.0,
  "download_timeout_seconds": 120.0,
  "number_of_download_processes": 1,
  "max_retries": null
}


#### coordenadas de los lagos

In [5]:
# coordenadas de los lagos que están en el lab 
lagos = {
    'Atitlán': {
        'west': -91.326256 ,
        'east': -91.07151 ,
        'south': 14.5948 ,
        'north': 14.750979
    },

    'Amatitlán': {
        'west': -90.638065 ,
        'east': -90.512924 ,
        'south': 14.412347 ,
        'north': 14.493799
    }
}

# fechas dadas en el lab
fechas_atitlan = [ '2025-01-18', '2025-04-13', '2025-05-13', '2025-07-17', '2025-11-21', '2025-12-29', '2026-02-12', '2026-03-24', '2026-04-13', '2026-04-28', '2026-07-22' ]

fechas_amatitlan = [ '2025-01-28', '2025-04-15', '2025-04-28', '2025-11-24', '2026-01-08', '2026-02-02', '2026-02-07', '2026-03-29', '2026-04-13', '2026-04-28', '2026-06-19' ]

fechas = { 'Atitlán': fechas_atitlan, 'Amatitlán': fechas_amatitlan }

print("Lagos definidos:")
for lago, coords in lagos.items(): print(f"  {lago}: {coords}")
print(f"\nFechas por lago:")
for lago, fecha_list in fechas.items(): print(f"  {lago}: {len(fecha_list)} fechas")

Lagos definidos:
  Atitlán: {'west': -91.326256, 'east': -91.07151, 'south': 14.5948, 'north': 14.750979}
  Amatitlán: {'west': -90.638065, 'east': -90.512924, 'south': 14.412347, 'north': 14.493799}

Fechas por lago:
  Atitlán: 11 fechas
  Amatitlán: 11 fechas


directorio para guardar datos

In [6]:
# Crear carpeta para guardar datos descargados
data_dir = Path('datos_sentinel')
data_dir.mkdir(exist_ok=True)

# Subcarpetas para cada lago
for lago in lagos.keys():
    lago_dir = data_dir / lago
    lago_dir.mkdir(exist_ok=True)

print(f"Directorio de datos: {data_dir.absolute()}")
print(f"Subdirectorios creados para cada lago")

Directorio de datos: c:\Users\Usuario Preinstalado\Documents\NoCuarentena.exe\Semestre 8\Data Science\DataS_Lab4_GeoEspaciales\datos_sentinel
Subdirectorios creados para cada lago


#### ddescargar datos de Sentinel-2

aquí bandas necesarias para calcular 
NDVI: B04 (Rojo) y B08 (NIR - Near Infrared)
NDWI: B03 (Verde) y B08 (NIR)
Cianobacteria



Descarga bandas específicas de Sentinel-2 para un lago en una fecha determinada.
    
    Args:
        bbox (BBox): Caja delimitadora con coordenadas del lago
        fecha (str): Fecha en formato 'YYYY-MM-DD'
        lago_nombre (str): Nombre del lago
        config: Configuración de Sentinel Hub
    
    Returns:
        dict: Diccionario con arrays NumPy de cada banda (B03, B04, B08)

In [7]:
def descargar_bandas_sentinelhub(bbox, fecha, lago_nombre, config=None):

    
   
    
    # las bandas B03 (verde), B04 (rojo), B08 (NIR)
    #  SCL para m de nubes
    request = SentinelHubRequest(
        evalscript="""
            //VERSION=3
            function setup() {
                return {
                    input: [{
                        bands: ["B03", "B04", "B08", "SCL"],
                        units: "DN"
                    }],
                    output: {
                        bands: 4,
                        sampleType: "FLOAT32"
                    }
                };
            }

            function evaluatePixel(sample) {
                return [sample.B03, sample.B04, sample.B08, sample.SCL];
            }
        """,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(fecha, fecha),
            )
        ],
        responses=[  SentinelHubRequest.output_response("default", MimeType.FLOAT32)  ],
        bbox=bbox,
        size=bbox_to_dimensions(bbox, resolution=10 ) ,
        config=config
    )
    
    try:
        data = request.get_data()
        print(f"decargado: {lago_nombre} y {fecha}")
        return data
    except Exception as e: 
        print(f"error para: {lago_nombre} y {fecha}: {str(e)}")
        return None

print("descarga lista")

descarga lista
